In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader


In [2]:
ds_gfs = xr.open_dataset('GFS_UPDATED_V1.nc')
ds_sla = xr.open_dataset('SLA_UPDATED.nc')


In [3]:
variables = [
    "sla",
    "TMP_surface",
    "UGRD_10maboveground",
    "VGRD_10maboveground",
    "PRATE_surface",
    "DSWRF_surface",
    "USWRF_surface",
    "DLWRF_surface",
    "ULWRF_surface",
    "SPFH_2maboveground"
]

In [6]:
ds = xr.merge([
    ds_sla[["sla"]],
    ds_gfs[variables[1:]]
])

In [7]:
target = ds['sla']

In [8]:
train_ds = ds.sel(time=slice(None, "2023-12-31"))

test_ds = ds.sel(time=slice("2024-01-01", None))

train_target = target.sel(time=slice(None, "2023-12-31"))

test_target = target.sel(time=slice("2024-01-01", None))

In [9]:
means = {}
stds = {}

for var in variables:

    means[var] = train_ds[var].mean(skipna=True)

    stds[var] = train_ds[var].std(skipna=True)

    print(
        f"{var:25}",
        f"mean={float(means[var]):10.4f}",
        f"std={float(stds[var]):10.4f}"
    )

sla                       mean=    0.0919 std=    0.0905
TMP_surface               mean=  301.4708 std=    5.0400
UGRD_10maboveground       mean=    1.1343 std=    4.0516
VGRD_10maboveground       mean=    0.2166 std=    3.7267
PRATE_surface             mean=    0.0000 std=    0.0001
DSWRF_surface             mean=  250.9850 std=   54.7321
USWRF_surface             mean=   27.6067 std=   21.4303
DLWRF_surface             mean=  395.3973 std=   39.2619
ULWRF_surface             mean=  467.5513 std=   30.8340
SPFH_2maboveground        mean=    0.0149 std=    0.0049


In [10]:
def normalize_dataset(ds, means, stds, variables):

    ds_norm = ds.copy()

    for var in variables:
        ds_norm[var] = (ds[var] - means[var]) / stds[var]

    return ds_norm

In [11]:
train_ds.values

<bound method Mapping.values of <xarray.Dataset> Size: 2GB
Dimensions:              (time: 3286, latitude: 116, longitude: 160)
Coordinates:
  * time                 (time) datetime64[ns] 26kB 2015-01-01 ... 2023-12-31
  * latitude             (latitude) float32 464B 0.125 0.375 ... 28.62 28.88
  * longitude            (longitude) float32 640B 50.12 50.38 ... 89.62 89.88
Data variables:
    sla                  (time, latitude, longitude) float32 244MB -0.0456 .....
    TMP_surface          (time, latitude, longitude) float32 244MB 300.9 ... ...
    UGRD_10maboveground  (time, latitude, longitude) float32 244MB -3.028 ......
    VGRD_10maboveground  (time, latitude, longitude) float32 244MB -6.598 ......
    PRATE_surface        (time, latitude, longitude) float32 244MB 1.719e-06 ...
    DSWRF_surface        (time, latitude, longitude) float32 244MB 292.7 ... ...
    USWRF_surface        (time, latitude, longitude) float32 244MB 19.34 ... ...
    DLWRF_surface        (time, latitude, l

In [12]:
ocean_mask = ~ds_sla.sla.isel(time=0).isnull()

In [13]:
train_ds_norm = normalize_dataset(train_ds, means, stds, variables)
test_ds_norm = normalize_dataset(test_ds, means, stds, variables)

train_target_norm = (train_target - means["sla"]) / stds["sla"]
test_target_norm = (test_target - means["sla"]) / stds["sla"]

In [14]:
print(train_ds["sla"].mean(skipna=True).item())
print(train_ds["sla"].std(skipna=True).item())

0.09185301512479782
0.09050951153039932


In [17]:
history = 14
output_window = 5

In [ ]:
n_samples = len(train_ds_norm.time) - history - output_window + 1

print(n_samples)

3268


In [19]:
len(ds_gfs.time)

3930

In [61]:
idx = 0

In [62]:
x = train_ds_norm.isel(
    time=slice(idx, idx + history)
)

In [63]:
y = train_target_norm.isel(
    time=idx + history - 1
)

In [64]:
class SLADataset(Dataset):

    def __init__(self,
                 ds,
                 target,
                 variables,
                 history,
                 forecast_days=5):

        self.ds = ds
        self.target = target
        self.variables = variables
        self.history = history
        self.forecast_days = forecast_days

        self.n_samples = (
            len(ds.time)
            - history
            - forecast_days
            + 2
        )

    def __len__(self):

        return self.n_samples

    def __getitem__(self, idx):

        x = self.ds.isel(
            time=slice(idx, idx + self.history)
        )

        y = self.target.isel(
            time=slice(
                idx + self.history - 1,
                idx + self.history - 1 + self.forecast_days
            )
        )

        x = np.stack(
            [x[var].values for var in self.variables],
            axis=1
        )

        y = y.values

        x = np.nan_to_num(x, nan=0.0)
        y = np.nan_to_num(y, nan=0.0)

        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)

        return x, y

In [65]:
train_dataset = SLADataset(
    train_ds_norm,
    train_target_norm,
    variables,
    history
)

test_dataset = SLADataset(
    test_ds_norm,
    test_target_norm,
    variables,
    history
)

In [66]:
len(train_dataset)

3269

In [67]:
x, y = train_dataset[0]

print(x.shape)
print(y.shape)

torch.Size([14, 10, 116, 160])
torch.Size([5, 116, 160])


In [68]:
print(torch.isnan(x).sum())
print(torch.isnan(y).sum())

tensor(0)
tensor(0)


In [69]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [70]:
test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [71]:
x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)

torch.Size([8, 14, 10, 116, 160])
torch.Size([8, 5, 116, 160])


In [72]:
conv = nn.Conv2d(
    in_channels=10,
    out_channels=32,
    kernel_size=3,
    padding=1
)

In [73]:
class CNNEncoder(nn.Module):

    def __init__(self,
                 in_channels=10,
                 hidden_channels=32):

        super().__init__()

        self.hidden_channels = hidden_channels

        self.encoder = nn.Sequential(

            nn.Conv2d(
                in_channels=in_channels,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.Conv2d(
                in_channels=16,
                out_channels=hidden_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU()

        )

    def forward(self, x):

        batch, time, channels, height, width = x.shape

        # Merge batch and time
        x = x.reshape(
            batch * time,
            channels,
            height,
            width
        )

        # CNN
        x = self.encoder(x)

        # Recover sequence dimension
        x = x.reshape(
            batch,
            time,
            self.hidden_channels,
            height,
            width
        )

        return x

In [74]:
encoder = CNNEncoder()

y = encoder(x)

print(y.shape)

torch.Size([8, 14, 32, 116, 160])


In [75]:
class ConvLSTMCell(nn.Module):

    def __init__(self,
                 input_channels=32,
                 hidden_channels=32):

        super().__init__()

        self.hidden_channels = hidden_channels

        self.conv = nn.Conv2d(
            input_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size=3,
            padding=1
        )

    def forward(self,
                x,
                h_prev,
                c_prev):

        combined = torch.cat(
            [x, h_prev],
            dim=1
        )

        gates = self.conv(combined)

        f, i, g, o = torch.chunk(
            gates,
            4,
            dim=1
        )

        f = torch.sigmoid(f)
        i = torch.sigmoid(i)
        g = torch.tanh(g)
        o = torch.sigmoid(o)

        c = f * c_prev + i * g

        h = o * torch.tanh(c)

        return h, c

In [76]:
class ConvLSTM(nn.Module):

    def __init__(self,
                 input_channels=32,
                 hidden_channels=32):

        super().__init__()

        self.hidden_channels = hidden_channels

        self.cell = ConvLSTMCell(
            input_channels,
            hidden_channels
        )

    def forward(self, x):

        batch, time, channels, height, width = x.shape

        h = torch.zeros(
            batch,
            self.hidden_channels,
            height,
            width,
            device=x.device
        )

        c = torch.zeros(
            batch,
            self.hidden_channels,
            height,
            width,
            device=x.device
        )

        for t in range(time):

            h, c = self.cell(
                x[:, t],
                h,
                c
            )

        return h

In [77]:
class Decoder(nn.Module):

    def __init__(self,
                 hidden_channels=32,
                 out_channels=5):

        super().__init__()

        self.conv = nn.Conv2d(
            hidden_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):

        return self.conv(x)

In [78]:
class ConvLSTMModel(nn.Module):

    def __init__(
        self,
        input_channels=10,
        hidden_channels=32,
        forecast_days=5
    ):

        super().__init__()

        self.encoder = CNNEncoder(
            in_channels=input_channels,
            hidden_channels=hidden_channels
        )

        self.convlstm = ConvLSTM(
            input_channels=hidden_channels,
            hidden_channels=hidden_channels
        )

        self.decoder = Decoder(
            hidden_channels=hidden_channels,
            out_channels=forecast_days
        )

    def forward(self, x):

        x = self.encoder(x)

        x = self.convlstm(x)

        x = self.decoder(x)

        return x

In [79]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ConvLSTMModel(
    input_channels=10,
    hidden_channels=32,
    forecast_days=5
).to(device)

In [80]:
x, y = next(iter(train_loader))

x = x.to(device)
y = y.to(device)

pred = model(x)

print("Prediction:", pred.shape)
print("Ground Truth:", y.shape)

Prediction: torch.Size([8, 5, 116, 160])
Ground Truth: torch.Size([8, 5, 116, 160])


In [82]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ConvLSTMModel(
    input_channels=10,
    hidden_channels=32,
    forecast_days=5
).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

In [ ]:
epochs = 30

best_val_loss = float("inf")   # <-- Initialize once, before the epoch loop

for epoch in range(epochs):

    model.train()

    train_loss = 0.0

    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        pred = model(x)

        loss = criterion(pred, y)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    
    #Validation

    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)
            y = y.to(device)

            pred = model(x)

            loss = criterion(pred, y)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(test_loader)
    
    #best model

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
        }, "best_convlstm.pth")

        print("✓ Best model saved!")
        

    scheduler.step(avg_val_loss)
    

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.6f} | "
        f"Validation Loss: {avg_val_loss:.6f}"
    )

✓ Best model saved!
Epoch 1/5 | Train Loss: 75.385739 | Validation Loss: 61.609981
✓ Best model saved!
Epoch 2/5 | Train Loss: 33.406028 | Validation Loss: 33.890329
✓ Best model saved!
Epoch 3/5 | Train Loss: 17.972253 | Validation Loss: 20.936051
✓ Best model saved!
Epoch 4/5 | Train Loss: 11.321066 | Validation Loss: 15.505256
✓ Best model saved!
Epoch 5/5 | Train Loss: 8.617516 | Validation Loss: 13.337025
